In [1]:
!pip install beautifulsoup4 -q

In [6]:
# ── CELL 1: MOUNT & INSTALL ────────────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive', force_remount=False)
# !pip install statsmodels -q
#!pip install regex
#import regex as re

ValueError: Mountpoint must not already contain files

In [5]:
import subprocess
result = subprocess.run(['sync'], capture_output=True)
print("Sync complete")

# Then wait a moment and check
import time
time.sleep(5)

# Force Drive to flush by accessing it
os.system('ls /content/drive/MyDrive/ > /dev/null')
print("Done — check Drive now")

Sync complete
Done — check Drive now


In [3]:
import os

# Check if the directories actually exist and are writable
JSON_DIR = '/content/drive/MyDrive/HEC Thesis/Text Data/structured_json_statements'
TXT_DIR  = '/content/drive/MyDrive/HEC Thesis/Text Data/fomc_statements'

print(f"JSON_DIR exists: {os.path.exists(JSON_DIR)}")
print(f"TXT_DIR exists:  {os.path.exists(TXT_DIR)}")

# Try writing a test file directly
test_path = os.path.join(JSON_DIR, 'test_write.json')
try:
    with open(test_path, 'w') as f:
        f.write('{"test": true}')
        f.flush()
        os.fsync(f.fileno())
    print(f"✓ Test file written successfully to Drive")
    print(f"  File exists: {os.path.exists(test_path)}")
    os.remove(test_path)
except Exception as e:
    print(f"✗ Write failed: {e}")

# Count existing files
print(f"\nExisting JSON files: {len(os.listdir(JSON_DIR))}")
print(f"First 5: {sorted(os.listdir(JSON_DIR))[:5]}")

JSON_DIR exists: True
TXT_DIR exists:  True
✓ Test file written successfully to Drive
  File exists: True

Existing JSON files: 200
First 5: ['statement_19940204.json', 'statement_19940322.json', 'statement_19940418.json', 'statement_19940517.json', 'statement_19940816.json']


In [ ]:
"""
scrape_fomc_statements.py
════════════════════════════════════════════════════════════════════════════════
Scrapes all FOMC press release statements from federalreserve.gov (1994–present)
and saves them as:
  1. Raw .txt files  →  /Text Data/fomc_statements/statement_YYYYMMDD.txt
  2. Structured JSON →  /Text Data/structured_json_statements/statement_YYYYMMDD.json

JSON schema:
  {
    "source":     "statement_YYYYMMDD.txt",
    "date":       "YYYY-MM-DD",
    "date_raw":   "YYYYMMDD",
    "type":       "fomc_statement",
    "text":       "...",
    "word_count": 440
  }

USAGE (Google Colab):
  exec(open('scrape_fomc_statements.py').read())

REQUIREMENTS:
  pip install requests beautifulsoup4
"""

import os
import re
import json
import time
import requests
from bs4 import BeautifulSoup
from pathlib import Path
import pandas as pd

# ════════════════════════════════════════════════════════════════════════════
# CONFIG
# ════════════════════════════════════════════════════════════════════════════

BASE_DRIVE    = '/content/drive/MyDrive/HEC Thesis'
TXT_DIR       = os.path.join(BASE_DRIVE, 'Text Data', 'fomc_statements')
JSON_DIR      = os.path.join(BASE_DRIVE, 'Text Data', 'structured_json_statements')

START_YEAR    = 1994
END_YEAR      = 2025   # inclusive

SLEEP_BETWEEN = 0.4    # seconds between requests (be polite to Fed servers)
SLEEP_YEAR    = 1.0    # seconds between year pages

FED_BASE      = 'https://www.federalreserve.gov'

# ── Create output directories ─────────────────────────────────────────────────
Path(TXT_DIR).mkdir(parents=True, exist_ok=True)
Path(JSON_DIR).mkdir(parents=True, exist_ok=True)
print(f"  TXT  → {TXT_DIR}")
print(f"  JSON → {JSON_DIR}")

# ════════════════════════════════════════════════════════════════════════════
# HELPERS
# ════════════════════════════════════════════════════════════════════════════

HEADERS = {
    'User-Agent': (
        'Mozilla/5.0 (Windows NT 10.0; Win64; x64) '
        'AppleWebKit/537.36 (KHTML, like Gecko) '
        'Chrome/120.0.0.0 Safari/537.36'
    )
}

def get_page(url, retries=3):
    """Fetch a URL with retries. Returns response text or None."""
    for attempt in range(retries):
        try:
            r = requests.get(url, headers=HEADERS, timeout=20)
            if r.status_code == 200:
                return r.text
            elif r.status_code == 404:
                return None   # page doesn't exist — not an error
            else:
                print(f"    HTTP {r.status_code} for {url}")
                time.sleep(2)
        except Exception as e:
            print(f"    Attempt {attempt+1} failed: {e}")
            time.sleep(2)
    return None


def extract_statement_text(html):
    """
    Extract clean statement body text from an FOMC press release page.
    Handles both modern and legacy Fed HTML formats.
    """
    if not html:
        return None

    soup = BeautifulSoup(html, 'html.parser')

    # ── Modern format (2011+): div.col-xs-12.col-sm-8.col-md-8 ──────────────
    content_div = (
        soup.find('div', class_='col-xs-12 col-sm-8 col-md-8') or
        soup.find('div', id='content') or
        soup.find('div', class_='article') or
        soup.find('div', id='leftText')
    )

    if content_div:
        # Remove navigation, scripts, styles
        for tag in content_div.find_all(['script', 'style', 'nav', 'footer']):
            tag.decompose()
        text = content_div.get_text(separator=' ', strip=True)
    else:
        # ── Legacy format (pre-2011): extract body text ───────────────────────
        body = soup.find('body')
        if body:
            for tag in body.find_all(['script', 'style', 'nav']):
                tag.decompose()
            text = body.get_text(separator=' ', strip=True)
        else:
            text = soup.get_text(separator=' ', strip=True)

    # ── Clean up whitespace ───────────────────────────────────────────────────
    text = re.sub(r'\s+', ' ', text).strip()

    # ── Extract the substantive statement body ────────────────────────────────
    # Try to find text between "For immediate release" / "For release" and footer
    body_patterns = [
        r'For (?:immediate )?release\s*(.*?)(?:Last [Uu]pdate|Board of Governors|'
        r'Home\s*\||\Z)',
        r'FEDERAL RESERVE\s+press release\s*(.*?)(?:Last [Uu]pdate|\Z)',
    ]
    for pat in body_patterns:
        m = re.search(pat, text, re.DOTALL | re.IGNORECASE)
        if m:
            body_text = m.group(1).strip()
            if len(body_text) > 100:
                return body_text

    # Fallback: return full cleaned text if patterns don't match
    if len(text) > 100:
        return text
    return None


def find_statement_urls_for_year(year):
    """
    Scrape the FOMC historical page for a given year and return a list of
    (date_str, statement_url) tuples.

    Two URL patterns exist:
      Modern  (2005+): /newsevents/pressreleases/monetaryYYYYMMDDa.htm
      Legacy  (1994-2004): /boarddocs/press/monetary/YYYY/YYYYMMDD/default.htm
                           or /fomc/YYYYMMDD/default.htm
    """
    # ── Try the modern historical page first ─────────────────────────────────
    urls_found = []

    year_url = f'{FED_BASE}/monetarypolicy/fomchistorical{year}.htm'
    html     = get_page(year_url)

    if html:
        soup = BeautifulSoup(html, 'html.parser')

        # Pattern 1: Modern press release links
        # /newsevents/pressreleases/monetary20040128a.htm
        for a in soup.find_all('a', href=True):
            href = a['href']
            m = re.search(
                r'/newsevents/pressreleases/monetary(\d{8})a\.htm',
                href, re.IGNORECASE
            )
            if m:
                date_str = m.group(1)
                full_url = FED_BASE + href if href.startswith('/') else href
                urls_found.append((date_str, full_url))

        # Pattern 2: Legacy boarddocs links (pre-2005)
        for a in soup.find_all('a', href=True):
            href = a['href']
            m = re.search(
                r'/boarddocs/press/monetary/\d{4}/(\d{8})/default\.htm',
                href, re.IGNORECASE
            )
            if m:
                date_str = m.group(1)
                full_url = FED_BASE + href if href.startswith('/') else href
                urls_found.append((date_str, full_url))

        # Pattern 3: Direct "Statement" link text
        if not urls_found:
            for a in soup.find_all('a', string=re.compile(r'Statement', re.I)):
                href = a.get('href', '')
                # Extract date from href if present
                m = re.search(r'(\d{8})', href)
                if m:
                    date_str = m.group(1)
                    full_url = FED_BASE + href if href.startswith('/') else href
                    urls_found.append((date_str, full_url))

    # ── For very old years (1994-1999) try alternative URL patterns ───────────
    if not urls_found and year <= 2000:
        # Try fetching the old-style monetary policy pages directly
        alt_url = f'{FED_BASE}/fomc/{year}/'
        html2 = get_page(alt_url)
        if html2:
            soup2 = BeautifulSoup(html2, 'html.parser')
            for a in soup2.find_all('a', href=True):
                href = a['href']
                m = re.search(r'(\d{8})', href)
                if m and 'statement' in a.get_text().lower():
                    date_str = m.group(1)
                    full_url = (FED_BASE + href if href.startswith('/')
                                else href)
                    urls_found.append((date_str, full_url))

    # Deduplicate keeping first occurrence per date
    seen = set()
    deduped = []
    for date_str, url in urls_found:
        if date_str not in seen:
            seen.add(date_str)
            deduped.append((date_str, url))

    return sorted(deduped)


def date_str_to_iso(date_str):
    """Convert '20040128' to '2004-01-28'."""
    return f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"


def save_statement(date_str, text):
    """Save statement to .txt and .json files."""
    filename_stem = f"statement_{date_str}"
    txt_path      = os.path.join(TXT_DIR,  f"{filename_stem}.txt")
    json_path     = os.path.join(JSON_DIR, f"{filename_stem}.json")

    # Save TXT
    with open(txt_path, 'w', encoding='utf-8') as f:
        f.write(text)

    # Save JSON
    record = {
        "source":     f"{filename_stem}.txt",
        "date":       date_str_to_iso(date_str),
        "date_raw":   date_str,
        "type":       "fomc_statement",
        "text":       text,
        "word_count": len(text.split()),
    }
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(record, f, indent=2, ensure_ascii=False)

    return record['word_count']


def already_scraped(date_str):
    """Check if this statement has already been saved."""
    json_path = os.path.join(JSON_DIR, f"statement_{date_str}.json")
    return os.path.exists(json_path)


# ════════════════════════════════════════════════════════════════════════════
# SPECIAL HANDLING: 1994-2004 legacy URL formats
# ════════════════════════════════════════════════════════════════════════════

# Pre-2005 statements are at different URL patterns.
# This dict provides known statement URLs for meetings that the scraper
# might miss due to non-standard page structures.
# Format: 'YYYYMMDD': 'full_url'

LEGACY_STATEMENT_URLS = {
    # 1994
    '19940204': f'{FED_BASE}/boarddocs/press/monetary/1994/19940204/default.htm',
    '19940322': f'{FED_BASE}/boarddocs/press/monetary/1994/19940322/default.htm',
    '19940517': f'{FED_BASE}/boarddocs/press/monetary/1994/19940517/default.htm',
    '19940706': f'{FED_BASE}/boarddocs/press/monetary/1994/19940706/default.htm',
    '19940816': f'{FED_BASE}/boarddocs/press/monetary/1994/19940816/default.htm',
    '19941115': f'{FED_BASE}/boarddocs/press/monetary/1994/19941115/default.htm',
    # 1995
    '19950201': f'{FED_BASE}/boarddocs/press/monetary/1995/19950201/default.htm',
    '19950307': f'{FED_BASE}/boarddocs/press/monetary/1995/19950307/default.htm',
    '19950706': f'{FED_BASE}/boarddocs/press/monetary/1995/19950706/default.htm',
    '19951219': f'{FED_BASE}/boarddocs/press/monetary/1995/19951219/default.htm',
    # Add more as needed — the scraper will find most automatically
}


# ════════════════════════════════════════════════════════════════════════════
# MAIN SCRAPING LOOP
# ════════════════════════════════════════════════════════════════════════════

print(f"\n{'═'*70}")
print(f"  FOMC Statement Scraper  |  {START_YEAR}–{END_YEAR}")
print(f"{'═'*70}\n")

all_results   = []
total_scraped = 0
total_skipped = 0
total_failed  = 0

for year in range(START_YEAR, END_YEAR + 1):
    print(f"\n── {year} ──────────────────────────────────────────────────────")

    # Get statement URLs for this year
    year_urls = find_statement_urls_for_year(year)

    # Supplement with legacy URLs for pre-2005 years
    if year <= 2004:
        legacy_for_year = {
            ds: url for ds, url in LEGACY_STATEMENT_URLS.items()
            if ds[:4] == str(year)
        }
        existing_dates = {ds for ds, _ in year_urls}
        for ds, url in legacy_for_year.items():
            if ds not in existing_dates:
                year_urls.append((ds, url))
        year_urls = sorted(year_urls)

    print(f"  Found {len(year_urls)} statement URLs")

    if not year_urls:
        print(f"  ⚠️  No URLs found for {year} — check manually")
        total_failed += 1
        time.sleep(SLEEP_YEAR)
        continue

    for date_str, url in year_urls:
        # Skip if already scraped
        if already_scraped(date_str):
            print(f"  ✓  {date_str_to_iso(date_str)}  [already exists — skip]")
            total_skipped += 1
            all_results.append({'date': date_str_to_iso(date_str),
                                 'status': 'skipped', 'url': url})
            continue

        # Fetch and parse
        html = get_page(url)
        text = extract_statement_text(html) if html else None

        if text and len(text) > 50:
            word_count = save_statement(date_str, text)
            print(f"  ✓  {date_str_to_iso(date_str)}  [{word_count} words]  {url}")
            total_scraped += 1
            all_results.append({'date': date_str_to_iso(date_str),
                                 'status': 'scraped',
                                 'word_count': word_count,
                                 'url': url})
        else:
            print(f"  ✗  {date_str_to_iso(date_str)}  FAILED  {url}")
            total_failed += 1
            all_results.append({'date': date_str_to_iso(date_str),
                                 'status': 'failed', 'url': url})

        time.sleep(SLEEP_BETWEEN)

    time.sleep(SLEEP_YEAR)

# ════════════════════════════════════════════════════════════════════════════
# SUMMARY
# ════════════════════════════════════════════════════════════════════════════

print(f"\n{'═'*70}")
print(f"  SCRAPING COMPLETE")
print(f"{'═'*70}")
print(f"  ✓  Scraped:  {total_scraped}")
print(f"  →  Skipped:  {total_skipped}  (already existed)")
print(f"  ✗  Failed:   {total_failed}")
print(f"  Total:       {total_scraped + total_skipped + total_failed}")

# Save manifest
results_df = pd.DataFrame(all_results)
manifest_path = os.path.join(BASE_DRIVE, 'Text Data', 'scrape_manifest.csv')
results_df.to_csv(manifest_path, index=False)
print(f"\n  Manifest saved → {manifest_path}")

# Show failed ones so you can manually fix
failed = results_df[results_df['status'] == 'failed']
if len(failed) > 0:
    print(f"\n  Failed statements (may need manual download):")
    for _, row in failed.iterrows():
        print(f"    {row['date']}  {row['url']}")

# Show word count distribution
if 'word_count' in results_df.columns:
    scraped = results_df[results_df['status'] == 'scraped']['word_count']
    if len(scraped) > 0:
        print(f"\n  Word count stats (scraped statements):")
        print(f"    Min:    {scraped.min():.0f}")
        print(f"    Median: {scraped.median():.0f}")
        print(f"    Max:    {scraped.max():.0f}")
        print(f"    Mean:   {scraped.mean():.0f}")

  TXT  → /content/drive/MyDrive/HEC Thesis/Text Data/fomc_statements
  JSON → /content/drive/MyDrive/HEC Thesis/Text Data/structured_json_statements

══════════════════════════════════════════════════════════════════════
  FOMC Statement Scraper  |  1994–2025
══════════════════════════════════════════════════════════════════════


── 1994 ──────────────────────────────────────────────────────
  Found 7 statement URLs
  ✓  1994-02-04  [99 words]  https://www.federalreserve.gov/fomc/19940204default.htm
  ✓  1994-03-22  [39 words]  https://www.federalreserve.gov/fomc/19940322default.htm
  ✓  1994-04-18  [36 words]  https://www.federalreserve.gov/fomc/19940418default.htm
  ✓  1994-05-17  [164 words]  https://www.federalreserve.gov/fomc/19940517default.htm
  ✗  1994-07-06  FAILED  https://www.federalreserve.gov/boarddocs/press/monetary/1994/19940706/default.htm
  ✓  1994-08-16  [200 words]  https://www.federalreserve.gov/fomc/19940816default.htm
  ✓  1994-11-15  [142 words]  https://www.fed

In [ ]:
import os
txt_dir  = '/content/drive/MyDrive/HEC Thesis/Text Data/fomc_statements'
json_dir = '/content/drive/MyDrive/HEC Thesis/Text Data/structured_json_statements'
print(f"TXT files:  {len(os.listdir(txt_dir))}")
print(f"JSON files: {len(os.listdir(json_dir))}")

TXT files:  200
JSON files: 200


In [ ]:
import os, json

json_dir = '/content/drive/MyDrive/HEC Thesis/Text Data/structured_json_statements'

files = sorted(os.listdir(json_dir))
print(f"Total files: {len(files)}")
print(f"First 10: {files[:10]}")
print(f"Last 5:   {files[-5:]}")

# Check for any pre-2011
pre2011 = [f for f in files if f < 'statement_2011']
print(f"\nPre-2011 files: {len(pre2011)}")
for f in pre2011[:5]:
    path = os.path.join(json_dir, f)
    with open(path) as fp:
        rec = json.load(fp)
    print(f"  {f}  →  date={rec['date']}  words={rec['word_count']}")

Total files: 200
First 10: ['statement_19940204.json', 'statement_19940322.json', 'statement_19940418.json', 'statement_19940517.json', 'statement_19940816.json', 'statement_19941115.json', 'statement_19950201.json', 'statement_19950706.json', 'statement_19951219.json', 'statement_19960131.json']
Last 5:   ['statement_20200729.json', 'statement_20200827.json', 'statement_20200916.json', 'statement_20201105.json', 'statement_20201216.json']

Pre-2011 files: 115
  statement_19940204.json  →  date=1994-02-04  words=99
  statement_19940322.json  →  date=1994-03-22  words=39
  statement_19940418.json  →  date=1994-04-18  words=36
  statement_19940517.json  →  date=1994-05-17  words=164
  statement_19940816.json  →  date=1994-08-16  words=200


In [ ]:
print(json_dir)  # prints the exact path the scraper saved to

/content/drive/MyDrive/HEC Thesis/Text Data/structured_json_statements
